<link rel="stylesheet" type="text/css" href="custom.css">

# Diete a confronto
## Peso, alimentazione, salute e fattori socioeconomici

### Sebastian Serafini <br> Primo anno di IBML - 2025/2026 

# Domande chiave:

* Quale dieta garantisce la maggior**perdita&nbsp;di&nbsp;peso**? Varia tra maschi e femmine?
* Come differisce l'apporto dei**macronutrienti**e di**calorie**tra le varie diete?
* Esistono correlazioni tra**parametri&nbsp;fisici**e perdita di peso?
* La scelta della dieta incide davvero sulla**salute**?
* C'è relazione tra il**reddito&nbsp;familiare**e il tipo di dieta seguita?


# Dataset: NHANES 2013-2014 
***(National Health and Nutrition Examination Survey)***

### L'indagine si basa sui dati raccolti dal**CDC&nbsp;statunitense**attraverso un questionario rivolto a oltre**10.000&nbsp;persone.**  <br><br> Il dataset è composto da moduli separati, che sono stati estratti, filtrati e uniti all'occorrenza:
### **demographic, labs, diet, medications, examination, questionnaire**


# Dieta e Dimagrimento
#### Quale dieta garantisce la maggior perdita di peso?

In [86]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Prima Domanda PRIMA PARTE

# lettura dei file CSV
diet = pd.read_csv('DataFrames/diet.csv')
exam = pd.read_csv('DataFrames/examination.csv')
quest = pd.read_csv('DataFrames/questionnaire.csv') 
demo = pd.read_csv('DataFrames/demographic.csv') 
labs = pd.read_csv('DataFrames/labs.csv')

# prova
#print("colonne in labs.csv:", labs.columns.tolist())

# unione dei dataframe (due a due), ricordando che la chiave rappresentante l'ID univoco nelle tabelle è 'SEQN'
df_merged = pd.merge(diet, exam, on='SEQN') # dataframe con tutte le colonne di diet e exam 
df_merged2 = pd.merge(df_merged, quest, on='SEQN') # dataframe con tutte le colonne di diet, exam e questionnaire
df_merged3 = pd.merge(df_merged2, demo, on='SEQN') # dataframe con tutte le colonne di diet, exam, questionnaire e demo
df = pd.merge(df_merged3, labs, on='SEQN')  # dataframe completo


# print("dimensioni iniziali del dataframe che ci serve: ", df.shape)


# identifico tutte le colonne relative al tipo di dieta (nel questionario NHANES la domanda sulle diete è a risposta multipla)
colonne_diete = [col for col in df.columns if col.startswith('DRQSDT')]

# filtro e rinonimo le colonne utili ricordando:
# SEQN = ID, DRQSDIET = on_diet?, WHD050 = peso_iniziale, BMXWT = peso_attuale, e devo sommare le colonne delle diete
colonne_utili = ['SEQN', 'DRQSDIET', 'WHD050', 'BMXWT'] + colonne_diete
df_utili = df[colonne_utili].copy()

df_renamed = df_utili.rename(columns={
    'SEQN' : 'ID',
    'DRQSDIET' : 'On_Diet?',
    'WHD050' : 'Peso_Iniziale',
    'BMXWT' : 'Peso_Attuale'
})


# filtro chi è a dieta, ossia chi ha il valore 1 nell'indice 'on_diet?'
df_filtered = df_renamed.loc[df_renamed['On_Diet?'] == 1.0].copy()
# display(df_filtered)

# creo la colonna unica per indicare il tipo di dieta (se uno ha indicato più diete, viene considerata quella col codice più basso)
df_filtered['Tipo_Dieta'] = df_filtered[colonne_diete].min(axis=1)

# mappiamo le diete usando il dizionario ufficiale con i codici delle rispettive diete
# creo il dizionario
dictionary_tipo_dieta = {
    1.0: 'Ipocalorica', 2.0: 'Povera di Grassi', 3.0: 'Iposodica', 4.0 :'Senza Zucchero', 
    5.0 : 'Povera di Fibre', 6.0 : 'Ricca di fibre', 7.0 : 'Diabetica', 8.0 : 'Sviluppo Muscolare',
    9.0 : 'Low-Carb', 10.0: 'Iperproteica', 11.0 : 'Senza Glutine', 12.0 : 'Renale', 91.0 : 'Altro'
}


# creo la colonna in fondo al dataframe che inizializzo su 'Altro' per gestire anche i NaN
df_filtered['Nome_Dieta'] = 'Altro'

# sostituisco i valori numerici della colonna riguardante il tipo di dieta con i nomi delle diete
for codice, nome in dictionary_tipo_dieta.items():
        df_filtered.loc[df_filtered['Tipo_Dieta'] == codice, 'Nome_Dieta'] = nome


# trasformo i codici errore in Peso Iniziale in NaN
df_filtered.loc[df_filtered['Peso_Iniziale'] == 7777, 'Peso_Iniziale'] = np.nan
df_filtered.loc[df_filtered['Peso_Iniziale'] == 9999, 'Peso_Iniziale'] = np.nan

# trasformazione dei codici errore o valori mancanti in BMXWT(indicati come '.') in NaN
df_filtered['Peso_Attuale'] == pd.to_numeric(df_filtered['Peso_Attuale'], errors='coerce') # forza la colonna a divnetare numerica e gestisce valori mancanti

# escludiamo la categoria 'Altro'
df_filtered = df_filtered[df_filtered['Nome_Dieta'] != 'Altro']

# converto il peso iniziale da libre a kg:
df_filtered['Peso_Iniziale'] = df_filtered['Peso_Iniziale'] * 0.453592

# calcolo i kg persi in 1 anno 
df_filtered['Kg_Persi'] = df_filtered['Peso_Iniziale'] - df_filtered['Peso_Attuale'] 
    

# pulizia di righe con dati mancanti NaN
df_filtered_clean = df_filtered.dropna(subset=['Nome_Dieta', 'Kg_Persi'])


# visualizzo il dataframe appena ripulito
colonne_visualizzate = ['ID', 'Nome_Dieta', 'Peso_Iniziale', 'Peso_Attuale', 'Kg_Persi']
df_clean = df_filtered_clean[colonne_visualizzate].copy()
# display(df_clean)


# FILTRAGGIO DIETE con meno pazienti
# trovo il numero di persone che segue ciascuna dieta:
num_persone_dieta = df_clean['Nome_Dieta'].value_counts()

# definisco quali sono le diete più seguite, trascuro quindi quelle con pochi pazienti (troppi pochi dati)
diete_ammesse = num_persone_dieta[num_persone_dieta >= 15].index

df_clean = df_clean[df_clean['Nome_Dieta'].isin(diete_ammesse)].copy()


#----------------------------------------------------------------------------
# VISUALIZZAZIONE GRAFICA: distribuzione dei kg persi per tipo di dieta (escludendo diete poco seguite)

# colore di sfondo di tutti i grafici: bianco
# sns.set(rc={'figure.facecolor': '#ffffff'})

# usando Seaborn
fig, ax = plt.subplots(figsize=(9,7), facecolor='#ffffff')
ax.set_facecolor('#ffffff')

# faccio la mediana dei Kg_Persi per ogni dieta, e ordino in modo decrescente dalla mediana maggiore a quella minore
# creo una Series dove l'indice è il nome della dieta e i valori sono le medie (media con mean)
mediane_diete = df_clean.groupby('Nome_Dieta')['Kg_Persi'].median()  # in Numpy avremmo usato np.median(array ecc..)

# verifica che è una series:
# print(type(mediane_diete))

# ordino la Series 
medie_ordinate = mediane_diete.sort_values(ascending=False)

# estraggo gli indici che abbiamo detto essere i nomi delle diete (ora ordinati)
ordine_mediane = medie_ordinate.index


# generazione del grafico boxplot, per confrontare le perdite di peso delle varie diete
sns.boxplot(
    data=df_clean,
    order=ordine_mediane,
    x='Kg_Persi',
    y='Nome_Dieta',
    hue='Nome_Dieta',
    palette='flare',
    legend=False,
    showfliers=False,
    width=0.6,
    boxprops={'alpha':0.8, 'edgecolor':'black', 'linewidth': 1},
    zorder=2  # box davanti ai pallini
);

sns.stripplot(
    x='Kg_Persi',
    y='Nome_Dieta',
    data=df_clean,
    order=ordine_mediane,
    jitter=True,
    color='grey',
    alpha=0.3,
    size=4,
    zorder=1, # dietro ai box
) # mostra i pallini

plt.axvline(0, color='red', linestyle='--', linewidth=1.5)  # linea rossa tratteggiata che mostra lo 0
ax.xaxis.grid(True, linestyle='--', alpha=0.6)  # griglia verticale
plt.xticks(np.arange(-30, 31, 5))  # numeri sull'asse delle x
plt.xlim(-30, 30)  # impostiamo un limite sull'asse x per evitare outlier 

plt.title('Distribuzione dei kg persi per tipo di dieta\n (1 anno)', fontsize=15, pad=15, weight='bold')
plt.xlabel('Chilogrammi (valore positivo = dimagrimento)', fontsize=15, labelpad=10)
# plt.ylabel('Tipo di dieta', fontsize=15)
plt.ylabel('')  # tolgo la scritta tipo di dieta
    
plt.tight_layout()

# salvataggio immagine del grafico
plt.savefig('img-grafici/grafico-slide-4.png', dpi=300, bbox_inches='tight', transparent=False)
plt.close()   # il grafico non viene mostrato


In [87]:
%%HTML
<div style="display: flex; align-items: center; justify-content: space-between; width: 100%;">
    <div style="flex: 0 0 55%; max-width: 55%;">
        <img src="img-grafici/grafico-slide-4.png" style="width: 100%; max-width: 100%; height: auto; object-fit: contain; display: block;">
    </div>

     <div style="flex: 0 0 40%; max-width: 40%;">
        <ul style="margin: 0; padding-left: 20px;">
            <li style="font-size: 18px; margin-bottom: 18px;">La dieta mirata allo<strong>'Sviluppo Muscolare'</strong>ha registrato il dimagrimento mediano più alto</li>
            <li style="font-size: 18px; margin-bottom: 18px;">La dieta<strong>'Ipocalorica'</strong>è la più seguita, ma presenta la<strong>variabilità più estrema</strong></li>
            <li style="font-size: 18px; margin-bottom: 18px;"><strong>Ampia dispersione</strong>dei dati: <br> sola scelta della dieta ≠ risultato garantito  </li>
        </ul>
    </div>
</div>


# Dieta e Dimagrimento
#### Varia tra maschi e femmine?

In [88]:
# Prima domanda SECONDA PARTE: l'efficacia della dieta cambia tra uomini e donne?
# Stiamo valutando un gruppo di pazienti più ampio, perchè ora vengon valutati il sesso, i kg persi e la dieta seguita

# recupero info dal dataframe di partenza 
df_genere_id = df[['SEQN', 'RIAGENDR']].rename(columns={
    'SEQN':'ID',
    'RIAGENDR': 'Codice_Genere'
}).copy()

# unione df_genere con df_clean creato all'inizio per avere Nome_Dieta e Kg_Persi
df_genere = pd.merge(df_clean, df_genere_id, on='ID', how='inner')

# creo il mini dizionario per mappare i due generi
gender_dictionary = {
    1.0: 'Uomini',
    2.0: 'Donne'
}

# creo la colonna in fondo al dataframe che inizializzo su 'Altro' per gestire anche i NaN
df_genere['Genere'] = 'Altro'

# sostituisco i valori numerici della colonna riguardante il genere
for codice, sesso in gender_dictionary.items():
        df_genere.loc[df_genere['Codice_Genere'] == codice, 'Genere'] = sesso

# escludiamo la categoria 'Altro'
df_genere = df_genere[df_genere['Genere'] != 'Altro']

# utilizziamo come ordinamento lo stesso usato per il primo grafico,

# VISUALIZZAZIONE GRAFICA: boxplot raggruppato

fig, ax = plt.subplots(figsize=(9,7), facecolor='#ffffff')
ax.set_facecolor('#ffffff')

sns.boxplot(
    data=df_genere,
    order=ordine_mediane,
    x='Kg_Persi',
    y='Nome_Dieta',
    hue='Genere',
    palette={'Uomini': 'blue', 'Donne': 'pink'},
    width=0.7,
    linewidth=1.0,
    showfliers=False,
    boxprops={'alpha':0.8, 'edgecolor':'black', 'linewidth': 1}
)


plt.axvline(0, color='red', linestyle='--', linewidth=1.5)  # linea rossa tratteggiata che mostra lo 0
plt.grid(True, linestyle='--', alpha=0.6)  # griglia verticale
plt.xticks(np.arange(-30, 31, 5))  # numeri sull'asse delle x
plt.xlim(-30, 30)  # limite sull'asse x per evitare outlier 

plt.title('Efficacia della dieta tra uomini e donne\n (1 anno)', fontsize=15, pad=15, weight='bold')
plt.xlabel('Chilogrammi (valore positivo = dimagrimento)', fontsize=15, labelpad=10)
# plt.ylabel('Tipo di dieta', fontsize=15)
plt.ylabel('')  # tolgo la scritta tipo di dieta

plt.tight_layout()

# salvataggio immagine del grafico
plt.savefig('img-grafici/grafico-slide-4.2.png', dpi=300, bbox_inches='tight', transparent=False)
plt.close()   # il grafico non viene mostrato


In [89]:
%%HTML
<div style="display: flex; align-items: center; justify-content: space-between; width: 100%;">
    <div style="flex: 0 0 55%; max-width: 55%;">
        <img src="img-grafici/grafico-slide-4.2.png" style="width: 100%; max-width: 100%; height: auto; object-fit: contain; display: block;">
    </div>

     <div style="flex: 0 0 40%; max-width: 40%;">
        <ul style="margin: 0; padding-left: 20px;">
            <li style="font-size: 18px; margin-bottom: 18px;">Gli<strong>uomini</strong>registrano generalmente una<strong>perdita</strong>di peso mediana<strong>superiore</strong> </li>
            <li style="font-size: 18px; margin-bottom: 18px;"><strong>'Ipocalorica'</strong>e<strong>'Senza Zucchero'</strong>mostrano il<strong>maggior calo ponderale</strong>per gli uomini </li>
            <li style="font-size: 18px; margin-bottom: 18px;"><strong>'Ipocalorica'</strong>e<strong>'Diabetica'</strong>mostrano<strong>maggior variabilità</strong>nei risultati </li>
        </ul>
    </div>
</div>
            


# Dieta e Macronutrienti
#### Come differisce l'apporto dei macronutrienti nelle varie diete?

In [90]:
# Seconda Domanda PRIMA PARTE: come differisce l'apporto dei macronutrienti tra le varie diete?
# Se voglio affidarmi ai valori dichiarati dai pazienti, dato che pulendo separatamente i NaN dei 
# macronutrienti e i NaN delle Kcal, si rischia di avere due campioni di pazienti diversi nei due grafici,
# devo pulire entrambi in un colpo solo

# Recuperiamo i dati riguardanti i macronutrienti e l'apporto calorico dal dataframe df creato prima
colonne_totali_id = ['SEQN', 'DR1TCARB', 'DR1TPROT', 'DR1TTFAT', 'DR1TKCAL']
df_utili_totali = df[colonne_totali_id].rename(columns={'SEQN':'ID'}).copy()

# Uniamo questi dati al df_filtered
df_clean_completo = pd.merge(df_filtered[['ID','Nome_Dieta']], df_utili_totali, on='ID')

# Rimuoviamo coloro che non hanno inserito i macronutrienti (NaN)
colonne_totali = ['DR1TCARB', 'DR1TPROT', 'DR1TTFAT', 'DR1TKCAL']
df_clean_completo = df_clean_completo.dropna(subset=colonne_totali)

# escludiamo la categoria 'Altro'
df_clean_completo = df_clean_completo[df_clean_completo['Nome_Dieta'] != 'Altro']

# filtriamo via le diete seguite da pochi
num_persone_dieta2 = df_clean_completo['Nome_Dieta'].value_counts()
diete_ammesse2 = num_persone_dieta2[num_persone_dieta2 >= 20].index
df_clean_completo = df_clean_completo[df_clean_completo['Nome_Dieta'].isin(diete_ammesse2)].copy()

# calcolo le medie dei macro per dieta
df_medie_macro = df_clean_completo.groupby('Nome_Dieta')[colonne_totali].mean().reset_index()


# ordiniamo e trasformiamo in percentuale (devo costruirla sulla base del'apporto calorico, non sul peso in grammi)
# calcolo le calorie derivanti da ogni macro ricordando:
# 4 kcal/g per Carbo e Prot, 9 kcal/g per Grassi
df_medie_macro['DR1TCARB'] = df_medie_macro['DR1TCARB'] * 4
df_medie_macro['DR1TPROT'] = df_medie_macro['DR1TPROT'] * 4
df_medie_macro['DR1TTFAT'] = df_medie_macro['DR1TTFAT'] * 9

# totale calorico dei macronutrienti
df_medie_macro['Totale'] = df_medie_macro['DR1TCARB'] + df_medie_macro['DR1TPROT'] + df_medie_macro['DR1TTFAT']

# ottengo la percentuale
df_perc_macro = df_medie_macro.copy()  # inizializzo il nuovo dataframe come copia del vecchio
df_perc_macro['DR1TCARB'] = (df_medie_macro['DR1TCARB'] / df_medie_macro['Totale']) * 100
df_perc_macro['DR1TPROT'] = (df_medie_macro['DR1TPROT'] / df_medie_macro['Totale']) * 100
df_perc_macro['DR1TTFAT'] = (df_medie_macro['DR1TTFAT'] / df_medie_macro['Totale']) * 100

# ordino dalla % di Carb più bassa alla pià alta
df_perc_macro = df_perc_macro.sort_values(by='DR1TCARB', ascending=True)
ordine_diete = df_perc_macro['Nome_Dieta'].tolist()

# rimuovo colonna Totale
df_perc_macro = df_perc_macro.drop(columns=['Totale'])

# rinonimo i valori dei macronutrienti
df_perc_macro = df_perc_macro.rename(columns={
    'DR1TCARB' : 'Carboidrati',
    'DR1TPROT' : 'Proteine',
    'DR1TTFAT' : 'Grassi'
})

# uniamo le tre colonne in una sola chiamata Macronutriente
df_perc_macro = df_perc_macro.melt(id_vars='Nome_Dieta', var_name='Macronutriente', value_name='Percentuale')

# diciamo che ordine usare (dato che histplot non ammette il parametro order)
df_perc_macro['Nome_Dieta'] = pd.Categorical(df_perc_macro['Nome_Dieta'], categories=ordine_diete, ordered=True)



# VISUALIZZAZIONE GRAFICA

plt.figure(figsize=(9,7), facecolor='#ffffff')
ax = plt.gca()  # per catturare asse ax
ax.set_facecolor('#ffffff')

ax = sns.histplot(
    data=df_perc_macro,
    x='Nome_Dieta',
    hue='Macronutriente',
    hue_order=['Grassi', 'Proteine', 'Carboidrati'],
    weights='Percentuale',
    multiple='stack',
    palette='flare',
    shrink=0.8)

# per inserire il numero percentuale nelle barre
for bar in ax.patches:
    height = bar.get_height()
    if height > 0:  # solo se l'altezza della barra è maggiore di 0
        ax.text(
            x = bar.get_x() + (bar.get_width() / 2), # centro orizzontale
            y = bar.get_y() + (height / 2),  # centro verticale
            s = f'{height:.1f}%',  # approssimo alla cifra decimale percentuale più vicino
            ha = 'center',
            va = 'center',
            color = 'white',
            fontsize = 10,
            weight = 'bold'
        )
        

plt.title('Apporto percentuale giornaliero di macronutrienti per ogni dieta', fontsize=15, pad=15, weight='bold')
# plt.xlabel('Tipo di dieta', fontsize=15, labelpad=10)
plt.xlabel('')
plt.ylabel('Percentuale', fontsize=15, labelpad=15)

# spostamento legenda
sns.move_legend(ax, loc='upper left', bbox_to_anchor=(1.05, 1))

# rotazione nomi sull'asse x
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.grid(axis='x', linestyle='')

plt.tight_layout()

# salvataggio immagine del grafico
plt.savefig('img-grafici/grafico-slide-5.png', dpi=300, bbox_inches='tight', transparent=False)
plt.close()   # il grafico non viene mostrato


In [91]:
%%HTML

<div style="display: flex; align-items: center; justify-content: space-between; width: 100%;">
    <div style="flex: 0 0 55%; max-width: 55%;">
        <img src="img-grafici/grafico-slide-5.png" style="width: 100%; max-width: 100%; height: auto; object-fit: contain; display: block;">
    </div>

     <div style="flex: 0 0 40%; max-width: 40%;">
        <ul style="margin: 0; padding-left: 20px;">
            <li style="font-size: 18px; margin-bottom: 18px;"> I<strong>carboidrati</strong>sono la fonte principale di quasi ogni dieta (>39%) </li>
            <li style="font-size: 18px; margin-bottom: 18px;"> La<strong>'Low-Carb'</strong>compensa il taglio dei carboidrati con i picchi di<strong>proteine</strong>e<strong>grassi</strong></li>
            <li style="font-size: 18px; margin-bottom: 18px;"> La dieta<strong>'Povera di Grassi'</strong>registra giustamente i<strong>lipidi minori</strong>(34.0%) </li> 
        </ul>
    </div>
</div>

# Dieta e Calorie
#### E l'apporto calorico?

In [92]:
# Seconda Domanda SECONDA PARTE: come differisce l'apporto calorico medio?
# Se voglio affidarmi ai valori dichiarati dai pazienti, dato che pulendo separatamente i NaN dei 
# macronutrienti e i NaN delle Kcal, si rischia di avere due campioni di pazienti diversi nei due grafici,
# devo pulire entrambi in un colpo solo (già fatto nella cella precedente)

# calcolo le medie dei kcal per dieta
df_medie_kcal = df_clean_completo.groupby('Nome_Dieta')['DR1TKCAL'].mean().reset_index()

# ordino le diete mantenendo l'ordine delle diete del primo grafico
df_medie_kcal['Nome_Dieta'] = pd.Categorical(df_medie_kcal['Nome_Dieta'], categories=ordine_diete, ordered=True)
df_medie_kcal = df_medie_kcal.sort_values('Nome_Dieta')


# VISUALIZZAZIONE GRAFICA

plt.figure(figsize=(9,7), facecolor='#ffffff')
ax = plt.gca()
ax.set_facecolor('#ffffff')

ax = sns.histplot(
    data=df_medie_kcal,
    x='Nome_Dieta',
    weights='DR1TKCAL',
    hue='Nome_Dieta',
    palette='flare', 
    shrink=0.8,
    legend=False)

# limiti sull'asse y per definire una partenza pià alta
plt.ylim(1500, 2500)

# per inserire il numero nelle barre
for bar in ax.patches:
    height = bar.get_height()
    if height > 0:  # solo se l'altezza della barra è maggiore di 0
        ax.text(
            x = bar.get_x() + (bar.get_width() / 2), # centro orizzontale
            y = bar.get_y() + ((1500+height) / 2),  # centro verticale
            s = f'{int(height)} \nkcal',  
            ha = 'center',
            va = 'center',
            color = 'black',
            fontsize = 10,
            weight = 'bold'
        )
        

plt.title('Apporto giornaliero medio di kcal per ogni dieta', fontsize=15, pad=15, weight='bold')
# plt.xlabel('Tipo di dieta', fontsize=15)
plt.xlabel('')  # tolgo la scitta tipo di dieta
plt.ylabel('KCal medie giornaliere', fontsize=15, labelpad=15)

# rotazione nomi sull'asse x
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.grid(axis='x', linestyle='')  # per togliere linee verticali

plt.tight_layout()

plt.savefig('img-grafici/grafico-slide-5.2.png', dpi=300, bbox_inches='tight', transparent=False)
plt.close()   # il grafico non viene mostrato


In [93]:
%%HTML

<div style="display: flex; align-items: center; justify-content: space-between; width: 100%;">
    <div style="flex: 0 0 55%; max-width: 55%;">
        <img src="img-grafici/grafico-slide-5.2.png" style="width: 100%; max-width: 100%; height: auto; object-fit: contain; display: block;">
    </div>

     <div style="flex: 0 0 40%; max-width: 40%;">
        <ul style="margin: 0; padding-left: 20px;">
            <li style="font-size: 18px; margin-bottom: 18px;"> La dieta per lo<strong>'Sviluppo Muscolare'</strong> registra<strong>l'apporto calorico</strong>più elevato</li>
            <li style="font-size: 18px; margin-bottom: 18px;"> I regimi<strong>'Senza Glutine'</strong>e<strong>'Diabetico'</strong>risultano i più<strong>restrittivi</strong>
            <li style="font-size: 18px; margin-bottom: 18px;"> Le altre diete si collocano tra le<strong>1820 e le 1950 kcal</strong>giornaliere </li> 
        </ul>
    </div>
</div>

# Parametri Fisici e Dimagrimento
#### Esistono correlazioni tra parametri fisici e pedita di peso?

In [94]:
# Terza domanda: esistono correlazioni tra parametri fisici e perdita di peso?
# Il problema è che i dati forniti sono quelli attuali, non quelli di 1 anno fa, perciò dobbiamo risalire ad essi tramite calcoli
# Dobbiamo quindi concentrarci sui parametri fisici a cui è possibile risalire, utilizzando ciò che è noto:
# età, altezza, peso, BMI

# recupero parametri fisici da examination.csv, sfrutto il dataframe df creato all'inizio
colonne_parametri_id = ['SEQN', 'WHD050', 'BMXWT', 'RIDAGEYR', 'BMXHT']
df_parametri = df[colonne_parametri_id].rename(columns={
    'SEQN':'ID',
    'WHD050':'Peso_Iniziale_lbs',
    'BMXWT':'Peso_Attuale',
    'BMXHT':'Altezza (cm)',
    'RIDAGEYR': 'Età'
}).copy()

# trasformo i codici errore in NaN
df_parametri.loc[df_parametri['Peso_Iniziale_lbs'] == 7777, 'Peso_Iniziale_lbs'] = np.nan
df_parametri.loc[df_parametri['Peso_Iniziale_lbs'] == 9999, 'Peso_Iniziale_lbs'] = np.nan

# trasformazione dei codici errore o valori mancanti in BMXWT(indicati come '.') in NaN
df_filtered['Peso_Attuale'] == pd.to_numeric(df_filtered['Peso_Attuale'], errors='coerce') # forza la colonna a divnetare numerica e gestisce valori mancanti

# rimozione NaN
df_parametri = df_parametri.dropna()

# converto il peso iniziale da libre a kg:
df_parametri['Peso_Iniziale'] = df_parametri['Peso_Iniziale_lbs'] * 0.453592

# calcolo i kg persi in 1 anno 
df_parametri['Kg_Persi'] = df_parametri['Peso_Iniziale'] - df_parametri['Peso_Attuale'] 


# Filtro solo i pazienti sopra i 22 anni (fino a 21 l'altezza può cambiare molto), voglio h costante
df_parametri = df_parametri[df_parametri['Età'] > 21].copy()

# trovo l'età iniziale
df_parametri['Età_Iniziale'] = df_parametri['Età'] - 1

# ricostruisco BMI dell'anno precedente, considerando come costante l'altezza
# formula BMI iniziale = peso iniziale / (altezza)^2
df_parametri['BMI_Iniziale'] = df_parametri['Peso_Iniziale']/((df_parametri['Altezza (cm)'] / 100) ** 2)

colonne_parametri = ['Età_Iniziale', 'Altezza (cm)', 'Peso_Iniziale', 'BMI_Iniziale', 'Kg_Persi']
df_correlazione = df_parametri[colonne_parametri]


# VISUALIZZAZIONE GRAFICA : Heatmap

plt.figure(figsize=(9,7), facecolor='#ffffff')
ax.set_facecolor('#ffffff')

matrice_corr = df_correlazione.corr()

mask = np.triu(np.ones_like(matrice_corr, dtype=bool))  # nascondere parte in alto a destra

# elimino la prima riga e l'ultima colonna (quelle con spazi vuoti)
matrice_corr = matrice_corr.iloc[1:, :-1]
mask = mask[1:, :-1]

sns.set_style('whitegrid')  # griglia

sns.heatmap(
    matrice_corr,
    mask=mask,
    annot=True,
    annot_kws={'size': 15, 'weight': 'bold'},  # aumentare grandezza numeri dentro heatmap
    cmap='flare',
    fmt='.2f',  # due cifre decimali
    vmin=-0.5,
    vmax=1,
    linewidths=.5,
    alpha=0.8
)


plt.title('Parametri fisici VS Perdita di Peso\n (Campioni adulti > 21 anni)', fontsize=15, pad=15, weight='bold')

plt.xticks(rotation=45, ha='right', fontsize=13)
plt.yticks(rotation=0, fontsize=13)


plt.tight_layout()

# salvataggio immagine del grafico
plt.savefig('img-grafici/grafico-slide-6.png', dpi=300, bbox_inches='tight', transparent=False)
plt.close()   # il grafico non viene mostrato 


In [95]:
%%HTML

<div style="display: flex; align-items: center; justify-content: space-between; width: 100%;">
    <div style="flex: 0 0 55%; max-width: 55%;">
        <img src="img-grafici/grafico-slide-6.png" style="width: 100%; max-width: 100%; height: auto; object-fit: contain; display: block;">
    </div>

     <div style="flex: 0 0 40%; max-width: 40%;">
        <ul style="margin: 0; padding-left: 15px;">
            <li style="font-size: 18px; margin-bottom: 18px;"> <strong>Non emergono</strong>forti correlazioni:</li>
            <ul style="margin: 0; padding-left: 30px;">
                <li style="font-size: 18px; margin-bottom: 18px;"> <strong>'BMI' e 'Peso Iniziale'</strong>sono <strong>debolmente legati</strong> ai Kg Persi</li>
                <li style="font-size: 18px; margin-bottom: 18px;"> <strong>'Età' e 'Altezza'</strong>risultano<strong>irrilevanti</strong></li>
            </ul>
        </ul>
    </div>
</div>

# Dieta e Rischio Metabolico
#### La scelta della dieta incide sulla salute?

In [96]:
# Quarta domanda: dieta e rischio metabolico; esiste un compromesso tra controllo glicemico e livelli di colesterolo
# nelle varie diete?

# bisogna trovare i valori di colesterolo totale e dell'emoglobina glicata, ossia la media di zuccheri negli scorsi 3 mesi, che sono:
# colesterolo totale: LBXTC, emoglobina glicata in %: LBXGH

colonne_labs_id = ['SEQN', 'LBXTC', 'LBXGH']

df_labs = df[colonne_labs_id].rename(columns={
    'SEQN':'ID',
    'LBXTC':'Colesterolo_Totale',
    'LBXGH':'Emoglobina_Glicata'
}).copy()

# unione con df_filtered (che non ha ancora rimosso i NaN)
df_esami = pd.merge(df_filtered[['ID', 'Nome_Dieta']], df_labs, on='ID', how='inner')

# rimozione dei NaN
df_esami = df_esami.dropna(subset=['Colesterolo_Totale', 'Emoglobina_Glicata', 'Nome_Dieta'])

# verifico che vada tutto 
num_persone_esami = df_esami['Nome_Dieta'].value_counts()
# print("Pazienti con analisi complete:", {len(df_esami)})  # 870
# display(numpersone__esami)
# display(df_esami.head())

# filtro via le diete poco significative come avevamo fatto per la prima domanda
diete_ammesse_esami = num_persone_esami[num_persone_esami >= 15].index
df_esami_clean = df_esami[df_esami['Nome_Dieta'].isin(diete_ammesse_esami)].copy()


# definisco l'ordine
mediane_esami = df_esami_clean.groupby('Nome_Dieta')['Emoglobina_Glicata'].median()

# ordino la Series 
esami_ordinati = mediane_esami.sort_values(ascending=False)

# estraggo gli indici che abbiamo detto essere i nomi delle diete (ora ordinati)
ordine_mediane_esami = esami_ordinati.index


# VISUALIZZAZIONE GRAFICA: doppio boxplot, uno a destra e uno a sinistra

# linee di soglia clinica (max):
# colesterolo totale: 200mg/dL
# emoglobina glicata: 5.7%

# PRIMO boxplot (sinistra)

plt.figure(figsize=(8,8), facecolor='#ffffff')
ax1 = plt.gca()  # per l'asse
ax1.set_facecolor('#ffffff')

sns.boxplot(
    data=df_esami_clean,
    x='Colesterolo_Totale',
    y='Nome_Dieta',
    hue='Nome_Dieta',
    order=ordine_mediane_esami,
    legend=False,
    ax=ax1,  # posiziono nel riquadro a sx
    palette='flare',
    width=0.6,
    linewidth=1.2,
    showfliers=False,
    boxprops={'alpha':0.8,}
)

# mettiamo in grassetto gli indici di colonna
plt.setp(ax1.get_yticklabels(), fontweight='bold', fontsize=15)

# inseriamo le linee di soglia clinica per il colesterolo
ax1.axvline(200, color='red', linestyle='--', linewidth=2.5, alpha=0.8)
ax1.text(220, 3.6, 'Soglia Clinica (>200)', color='red', fontsize=12, weight='bold')

ax1.set_title('Colesterolo Totale [mg/dL]', fontsize=15, weight='bold', pad=20)
ax1.set_xlabel('')
# ax1.set_ylabel('Tipo di dieta', fontsize=15, labelpad=10)
ax1.set_ylabel('') # togliere la scitta del tipo di dieta
ax1.grid(axis='x', linestyle='--', alpha=0.4)

# limiti sull'asse x per zoomare
plt.xlim(0, 400)

plt.tight_layout()

# salvataggio immagine del grafico
plt.savefig('img-grafici/grafico-slide-7-colesterolo.png', dpi=300, bbox_inches='tight', transparent=False)
plt.close()   # il grafico non viene mostrato 



# SECONDO boxplot (destra)

plt.figure(figsize=(8,8), facecolor='#ffffff')
ax2 = plt.gca()  # per l'asse
ax2.set_facecolor('#ffffff')

sns.boxplot(
    data=df_esami_clean,
    x='Emoglobina_Glicata',
    y='Nome_Dieta',
    hue='Nome_Dieta',
    order=ordine_mediane_esami,
    legend=False,
    ax=ax2,  # posiziono nel riquadro a dx
    palette='crest',
    width=0.6,
    linewidth=1.2,
    showfliers=False,
    boxprops={'alpha':0.8,}
)

# mettiamo in grassetto gli indici di colonna
plt.setp(ax2.get_yticklabels(), fontweight='bold', fontsize=15)

# inseriamo le linee di soglia clinica per l'emoglobina glicata
ax2.axvline(5.7, color='red', linestyle='--', linewidth=2.5, alpha=0.8)
ax2.text(6.2, 3.6, 'Soglia Clinica (>5.7%)', color='red', fontsize=12, weight='bold')

ax2.set_title('Emoglobina Glicata [%]', fontsize=15, weight='bold', pad=20)
ax2.set_xlabel('')
ax2.set_ylabel('') 
ax2.grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()

# salvataggio immagine del grafico
plt.savefig('img-grafici/grafico-slide-7-glicemia.png', dpi=300, bbox_inches='tight', transparent=False)
plt.close()   # il grafico non viene mostrato 


In [97]:
%%HTML

<div style="display: flex; flex-direction: column; align-items: center,  width: 100%; justify-content: center;">
          
   <!-- immagini dei due grafici affiancate -->
    <div style="display: flex; align-items: center; justify-content: center; gap: 5%; width: 87%;">
      
        <div style="flex: 1; margin-left: 115px">
            <img src="img-grafici/grafico-slide-7-colesterolo.png" style="width: 100% ; max-width: 100%; height: auto; object-fit: contain; display: inline-block; border-radius: 10px;">
        </div>
        
        <div style="flex: 1;">
            <img src="img-grafici/grafico-slide-7-glicemia.png" style="width: 100% ; max-width: 100% ;height: auto; object-fit: contain; display: inline-block; border-radius: 10px;">
        </div>
    </div>

    <!-- testi sotto -->
     <div style="width: 85%;">
        <ul style="margin: 0; margin-left: 120px;">
            <li style="font-size: 18px; margin-bottom: 5px;"> Il regime<strong>'Diabetico'</strong>presenta il valore più<strong>estremo</strong>di<strong>emoglobina glicata</strong></li>
            <li style="font-size: 18px; margin-bottom: 5px;"> Le diete<strong>'Senza Glutine' e 'Sviluppo Muscolare'</strong>registrano i<strong>migliori valori</strong></li>
        </ul></li>
    </div>
</div>

# Dieta e Reddito
#### Esiste una relazione tra reddito familiare e dieta seguita?

In [98]:
# Quinta domanda: c'è relazione tra il reddito e il tipo di dieta seguita? qual è la dieta seguita maggiormente dai ricchi?

# le info sul reddito sono all'interno di demographic.csv, il reddito INDFMPIR indica il rapporto tra reddito della famiglia e soglia di povertà americana
# 0 < x <= 1 : nella soglia di povertà
# x > 1: sopra la soglia di povertà
# x = 5: famiglia molto ricca

# recupero le info dal databsae df
colonne_reddito_id = ['SEQN', 'INDFMPIR']

df_reddito_id = df[colonne_reddito_id].rename(columns={
    'SEQN':'ID',
    'INDFMPIR': 'Indice_Reddito'
}).copy()

# unione con diete
df_reddito = pd.merge(df_filtered[['ID', 'Nome_Dieta']], df_reddito_id, on='ID', how='inner')

# rimozione dei NaN
df_reddito = df_reddito.dropna(subset=['Indice_Reddito', 'Nome_Dieta'])

# verifico che vada tutto 
num_persone_reddito = df_reddito['Nome_Dieta'].value_counts()
# print("Pazienti con reddito dichiarato:", {len(df_reddito)})  # 870
# display(num_persone_reddito)
# display(df_reddito.head())

# filtro via le diete poco significative come avevamo fatto per la prima domanda
diete_ammesse_reddito = num_persone_reddito[num_persone_reddito >= 15].index
df_reddito_clean = df_reddito[df_reddito['Nome_Dieta'].isin(diete_ammesse_reddito)].copy()


# definisco l'ordine
medie_redditi = df_reddito_clean.groupby('Nome_Dieta')['Indice_Reddito'].mean()

# ordino la Series 
redditi_ordinati = medie_redditi.sort_values(ascending=False)

# estraggo gli indici che abbiamo detto essere i nomi delle diete (ora ordinati)
ordine_medie_reddito = redditi_ordinati.index



# VISUALIZZAZONE GRAFICA: barplot

plt.figure(figsize=(15,7), facecolor='#ffffff')
ax = plt.gca()
ax.set_facecolor('#ffffff')

sns.barplot(
    data=df_reddito_clean,
    x='Indice_Reddito',
    y='Nome_Dieta',
    hue='Nome_Dieta',
    order=ordine_medie_reddito,
    palette='flare', 
    edgecolor='black',
    alpha=0.8,
    linewidth=0.5
)

# mettiamo in grassetto gli indici di colonna
plt.setp(ax.get_yticklabels(), fontweight='bold', fontsize=15)

plt.title('Relazione tra reddito familiare e dieta seguita', fontsize=20, pad=15, weight='bold')
plt.xlabel('Indice di Reddito (0 = minimo, 5 = massimo)', fontsize=18, labelpad=15)
# plt.ylabel('Tipo di Dieta', fontsize=15, labelpad=10)
plt.ylabel('')  # tolgo la scritta tipo di dieta

# linea di soglia
plt.axvline(1.0, color='orange', linestyle='--', linewidth=2.5)
plt.text(1.1, 1.1, 'Soglia limite (1.0)', color='orange', fontsize=20, weight='bold')
    
plt.grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()

# salvataggio immagine del grafico
plt.savefig('img-grafici/grafico-slide-8.png', dpi=300, bbox_inches='tight', transparent=False)
plt.close()   # il grafico non viene mostrato 

In [99]:
%%HTML

<div style="display: flex; align-items: center; flex-direction: column; width: 100%; gap: 5px, justify-content: center;">
    <div style="width: 80%;">
        <img src="img-grafici/grafico-slide-8.png" style="width: 100%; max-width: 100%; height: 100%; object-fit: contain; display: block;">
    </div>

     <div style="width: 75% !important">
        <ul style="margin: 0; padding-left: 10px;">
            <li style="font-size: 18px; margin-bottom: 5px;">La dieta<strong>'Senza Glutine'</strong>è seguita dalle famiglie con<strong>reddito più elevato</strong>(3.3)</li>
            <li style="font-size: 18px; margin-bottom: 5px;">La dieta per lo<strong>'Sviluppo Muscolare'</strong>non richiede un reddito alto (2.1) </li>
        </ul>
    </div>
</div>

<section id="conclusioni">

# Conclusioni sui dati analizzati 
<br> <br>
* Nessuna dieta garantisce il dimagrimento, ma il **genere** incide. <br>
* I macronutrienti sono simili tra le diete, conta il **bilancio&nbsp;calorico**.<br>
* Parametri fisici di partenza **influiscono&nbsp;debolmente** sull'efficacia di una dieta.<br>
* L'alimentazione incide sul **rischio&nbsp;metabolico**, diete apparentemente sane possono rivelarsi dannose.<br>
* Più una dieta è **strutturata**, **maggiore** è il **potere&nbsp;d'acquisto** richiesto.
</section>